# Clause-aware MiniLM + FAISS and abstention

Iterations 15 and 16 isolate chunking from a query-level cosine cutoff. Kanon was prepared but not run and is excluded from performance comparisons. This notebook reviews full-train saved results; it never evaluates test data.

In [2]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'src/retrieval.py').exists())
sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT/'scripts'))
from project_paths import source_path
rows=[json.loads(line) for line in (ROOT/'experiments/semantic_log.jsonl').read_text(encoding='utf-8').splitlines()]
clause=next(r for r in reversed(rows) if r['iteration']==15)
gate=next(r for r in reversed(rows) if r['iteration']==16)
clause['config']

{'name': 'dense_minilm_clause_faiss',
 'retriever_type': 'dense',
 'model_id': 'sentence-transformers/all-MiniLM-L6-v2',
 'revision': '1110a243fdf4706b3f48f1d95db1a4f5529b4d41',
 'dimension': 384,
 'max_seq_length': 256,
 'device': 'cpu',
 'torch_threads': 4,
 'batch_size': 32,
 'chunk_size': 700,
 'chunk_overlap': 0,
 'min_score': 0.0,
 'k_values': [1, 3, 5, 10],
 'query_mode': 'full',
 'normalize_embeddings': True,
 'faiss_index': 'IndexFlatIP',
 'chunk_strategy': 'clause_v1'}

## Boundaries and original offsets

Paragraph and numbered-section boundaries are preferred. Sentence units are packed up to 700 characters; oversized units split at whitespace. This is a deterministic heuristic, not a legal clause parser. No overlap or gold labels are used.

In [3]:
from clause_chunking import clause_chunks
text='1. Term. The agreement expires in 2030.\n\n2. Payments. Fees are payable monthly.'
pd.DataFrame(clause_chunks({'example':text},max_chars=70))

,file_path,start,end,text
0,example,0,41,1. Term. The agreement expires in 2030.\n\n
1,example,41,79,2. Payments. Fees are payable monthly.


In [4]:
table=[]
for name,result in clause['metrics'].items():
    metric=next(m for m in result['retrieval'] if m['k']==5)
    table.append(dict(version=name,precision=metric['precision'],recall=metric['recall'],benchmark_precision=metric['benchmark_precision'],benchmark_recall=metric['benchmark_recall'],false_abstention=result['false_abstention'],correct_abstention=result['correct_abstention']))
pd.DataFrame(table).set_index('version').style.format('{:.2%}')

,precision,recall,benchmark_precision,benchmark_recall,false_abstention,correct_abstention
version,,,,,,
reference,4.03%,35.75%,4.19%,38.90%,0.00%,0.00%
candidate,5.12%,19.74%,5.12%,19.74%,0.00%,0.00%


## Abstention sweep

Reject the entire query when its best similarity score does not exceed the threshold. Similarity is not a probability. Threshold selection uses train only, so these are tuning results, not independent validation estimates. Every attempted threshold remains recorded.

In [5]:
trials=[]
for trial in gate['trials']:
    m=next(m for m in trial['metrics']['retrieval'] if m['k']==5)
    trials.append(dict(threshold=trial['threshold'],precision=m['precision'],recall=m['recall'],false_abstention=trial['metrics']['false_abstention'],correct_abstention=trial['metrics']['correct_abstention'],accepted=trial['accepted']))
pd.DataFrame(trials)

,threshold,precision,recall,false_abstention,correct_abstention,accepted
0,0.2,0.051159,0.197358,0.000000,0.000000,False
1,0.3,0.051159,0.197358,0.000000,0.000000,False
2,0.4,0.051159,0.197358,0.000000,0.000000,False
3,0.5,0.051159,0.197358,0.000000,0.043478,True
4,0.6,0.049083,0.190215,0.071429,0.608696,False
5,0.7,0.019062,0.071173,0.600000,0.956522,False
6,0.8,0.000000,0.000000,1.000000,0.956522,False


In [6]:
print('Selected threshold:',gate['selected_threshold'])
print('Chunking checks:',clause['checks'])
print('Test evaluations:',clause['test_evaluation_runs'])

Selected threshold: 0.5
Chunking checks: {'precision_gain': True, 'recall': False, 'false_abstention': True, 'correct_abstention': True}
Test evaluations: 0


## Reproduce or modify

Change `semantic_configs/minilm_clause.json` for a new explicitly logged chunking version. `python manage.py evaluate_clause` reruns the full train evaluation and threshold sweep using cached vectors where available. Prior configs and results remain intact. Future definitions/cross-reference enrichment and hybrid retrieval are separate steps.